# Parakeet STT — Audio & Microphone Testing

Working notebook for:
- **Audio testing**: baseline + chunked streaming transcription on files
- **Microphone testing**: real-time STT using `sounddevice` (macOS/Linux friendly)

Uses **nvidia/parakeet-rnnt-1.1b** via NeMo. GPU recommended for real-time mic.

## 1. Install Dependencies

Run once. Skip if already installed.

In [ ]:
%%capture install_log
import subprocess, sys

# Skip apt-get (needs root). Install system libs manually if needed:
#   Ubuntu/Debian: sudo apt install sox libsndfile1 ffmpeg
#   macOS: brew install sox libsndfile ffmpeg

# Ensure setuptools/wheel available (fixes "setup.py" build errors)
subprocess.run([
    sys.executable, '-m', 'pip', 'install', '-q', '--upgrade',
    'pip', 'setuptools', 'wheel',
], check=True)

# Pin packaging (lightning/NeMo break with packaging 26+)
subprocess.run([
    sys.executable, '-m', 'pip', 'install', '-q',
    'packaging>=20.0,<25.0',
], check=True)

# Install Python packages
subprocess.run([
    sys.executable, '-m', 'pip', 'install', '-q',
    'librosa', 'soundfile', 'sounddevice', 'numpy<2',
], check=True)

subprocess.run([
    sys.executable, '-m', 'pip', 'install', '-q',
    'nemo_toolkit[asr]',
], check=True)

print('Installation complete.')

## 2. Imports & Environment

In [ ]:
import os

# Cache models in project folder (survives kernel restart, no re-download)
MODEL_CACHE = os.path.join(os.getcwd(), 'models')
os.makedirs(MODEL_CACHE, exist_ok=True)
os.environ['HF_HOME'] = MODEL_CACHE
os.environ['HUGGINGFACE_HUB_CACHE'] = os.path.join(MODEL_CACHE, 'hub')
os.environ['TORCH_HOME'] = os.path.join(MODEL_CACHE, 'torch')  # Silero VAD
print(f'Model cache: {MODEL_CACHE}')

import time
import urllib.request
import numpy as np
import torch
import librosa
import soundfile as sf
import nemo.collections.asr as nemo_asr

SAMPLE_RATE = 16000
device = 'cuda' if torch.cuda.is_available() else 'cpu'

print(f'PyTorch: {torch.__version__}')
print(f'CUDA: {torch.cuda.is_available()}')
if torch.cuda.is_available():
    print(f'GPU: {torch.cuda.get_device_name(0)}')

import nemo
print(f'NeMo: {nemo.__version__}')

## 3. Load Model

In [ ]:
# Use parakeet-tdt (faster, more reliable than rnnt)
MODEL_NAME = 'nvidia/parakeet-tdt-1.1b'

print(f'Loading {MODEL_NAME}...')
t0 = time.time()
asr_model = nemo_asr.models.EncDecRNNTBPEModel.from_pretrained(model_name=MODEL_NAME)
asr_model = asr_model.to(device)
asr_model.eval()
asr_model.freeze()
print(f'Loaded in {time.time()-t0:.1f}s on {device}')

## 4. Helper: Extract Text from RNNT Output

NeMo RNNT returns nested structures; this handles all variants.

In [ ]:
def extract_text(output):
    if isinstance(output, tuple) and len(output) > 0:
        output = output[0]
    if not isinstance(output, (list, tuple)) or len(output) == 0:
        return str(output)
    first = output[0]
    if isinstance(first, list) and first:
        first = first[0]
    return first.text if hasattr(first, 'text') else str(first)

## 5. Audio Testing — Download Sample

In [ ]:
AUDIO_URL = 'https://dldata-public.s3.us-east-2.amazonaws.com/2086-149220-0033.wav'
AUDIO_PATH = 'sample_audio.wav'

if not os.path.exists(AUDIO_PATH):
    print('Downloading sample...')
    urllib.request.urlretrieve(AUDIO_URL, AUDIO_PATH)

audio, sr = librosa.load(AUDIO_PATH, sr=SAMPLE_RATE, mono=True)
duration = len(audio) / SAMPLE_RATE
print(f'Audio: {AUDIO_PATH}, {duration:.2f}s, {len(audio):,} samples')

**Debug** (if baseline is still empty): Run below to inspect output structure.

## 7b. Optimized Streaming (VAD + TTFW)

Uses **Silero VAD** to skip silence and **variable first chunk** (1s) for faster time-to-first-word (~1.1s vs ~2.1s).

In [ ]:
from stt_optimized import (
    load_silero_vad,
    streaming_transcribe_optimized,
)

# Load Silero VAD (first run downloads ~2MB)
vad_model, vad_utils = load_silero_vad()
get_speech_timestamps = vad_utils[0]

# Optimized: 1s first chunk (TTFW ~1.1s), 2s rest, VAD skips silence
opt_text, opt_lat, ttfw_ms = streaming_transcribe_optimized(
    audio, asr_model, SAMPLE_RATE,
    first_chunk_s=1.0,
    rest_chunk_s=2.0,
    overlap_s=0.25,
    use_vad=True,
    use_fp16=False,
    vad_model=vad_model,
    get_speech_timestamps=get_speech_timestamps,
    device=device,
    verbose=True,
)

print(f'\nOptimized result: "{opt_text}"')
print(f'TTFW: {ttfw_ms:.0f} ms  |  Avg latency: {np.mean(opt_lat):.0f} ms')

## STT Blindspots & Optional Enhancements

| Component | Purpose | Status |
|-----------|---------|--------|
| **VAD** | Skip silence, cleaner output | ✅ Implemented (7b) |
| **Variable first chunk** | Faster TTFW | ✅ Implemented (7b) |
| **Punctuation restoration** | Add `.` `,` `?` to raw text | Optional: `pip install pyctcdecode` or NeMo ITN |
| **Inverse text normalization** | "twenty twenty five" → "2025" | NeMo has `nemo_text_processing` |
| **Noise reduction** | Pre-process noisy audio | Optional: `noisereduce`, `demucs` |
| **Speaker diarization** | "Who said what when" | Optional: `pyannote.audio` |
| **Echo cancellation** | Mic near speakers | WebRTC AEC, speex |
| **Model warmup** | First inference slower | ✅ Done in Section 6 |

In [ ]:
# Uncomment to inspect transcribe output structure:
# print("type(output):", type(output))
# print("len:", len(output) if hasattr(output, '__len__') else 'N/A')
# if hasattr(output, '__len__') and len(output) > 0:
#     print("output[0] type:", type(output[0]))
#     print("output[0] repr:", repr(output[0])[:300])
#     if hasattr(output[0], 'text'):
#         print("output[0].text:", repr(output[0].text))

## 6. Audio Testing — Baseline (Full-File)

In [ ]:
if device == 'cuda':
    warmup_audio = np.zeros(SAMPLE_RATE, dtype=np.float32)
    sf.write('/tmp/_warmup.wav', warmup_audio, SAMPLE_RATE)
    _ = asr_model.transcribe(['/tmp/_warmup.wav'], batch_size=1, verbose=False)
    torch.cuda.synchronize()
    print('Warmup done.')

if device == 'cuda':
    torch.cuda.synchronize()
t0 = time.perf_counter()
output = asr_model.transcribe([AUDIO_PATH], batch_size=1, verbose=False)
if device == 'cuda':
    torch.cuda.synchronize()
t1 = time.perf_counter()

baseline_text = extract_text(output)
latency_ms = (t1 - t0) * 1000
rtf = (t1 - t0) / duration

print(f'Baseline: "{baseline_text}"')
print(f'Latency: {latency_ms:.0f} ms, RTFx: {1/rtf:.0f}x')

## 7. Audio Testing — Chunked Streaming

In [ ]:
def streaming_transcribe(audio_arr, model, sr, chunk_s=2.0, overlap_s=0.5):
    chunk_n = int(chunk_s * sr)
    overlap_n = int(overlap_s * sr)
    step_n = chunk_n - overlap_n
    total = len(audio_arr)
    texts, latencies = [], []
    pos, idx = 0, 0

    while pos < total:
        end = min(pos + chunk_n, total)
        chunk = audio_arr[pos:end]
        if len(chunk) < int(0.1 * sr):
            break

        tmp = f'/tmp/_chunk_{idx}.wav'
        sf.write(tmp, chunk.astype(np.float32), sr)
        if device == 'cuda':
            torch.cuda.synchronize()
        t0 = time.perf_counter()
        result = model.transcribe([tmp], batch_size=1, verbose=False)
        if device == 'cuda':
            torch.cuda.synchronize()
        t1 = time.perf_counter()

        text = extract_text(result).strip()
        texts.append(text)
        lat_ms = (t1 - t0) * 1000
        latencies.append(lat_ms)

        print(f'[{idx:02d}] {pos/sr:.2f}s-{end/sr:.2f}s  {lat_ms:.0f}ms  | {text}')
        try:
            os.remove(tmp)
        except OSError:
            pass
        pos += step_n
        idx += 1

    return ' '.join(texts), latencies

stream_text, stream_lat = streaming_transcribe(audio, asr_model, SAMPLE_RATE, 2.0, 0.5)
print(f'\nStreaming result: "{stream_text}"')
print(f'Avg latency: {np.mean(stream_lat):.0f} ms')

## 8. Microphone Testing — Setup

Uses **sounddevice**. **Requires a physical microphone** — will fail in Docker/Colab/headless environments.

In [ ]:
import queue
import sounddevice as sd

CHUNK_SEC = 1.0
OVERLAP_SEC = 0.2

class FrameBuffer:
    def __init__(self, sr, chunk_sec, overlap_sec):
        self.sr = sr
        self.chunk_len = int(chunk_sec * sr)
        self.overlap_len = int(overlap_sec * sr)
        self.buffer = np.zeros(self.chunk_len + self.overlap_len, dtype=np.float32)
        self.filled = 0

    def add(self, audio):
        needed = self.chunk_len + self.overlap_len - self.filled
        if len(audio) < needed:
            self.buffer[self.filled:self.filled+len(audio)] = audio
            self.filled += len(audio)
            return None
        self.buffer[self.filled:self.filled+needed] = audio[:needed]
        out = self.buffer.copy()
        self.buffer[:self.overlap_len] = self.buffer[-self.overlap_len:]
        self.filled = self.overlap_len
        return out

buf = FrameBuffer(SAMPLE_RATE, CHUNK_SEC, OVERLAP_SEC)
audio_q = queue.Queue()

def sd_callback(indata, frames, time_info, status):
    if status:
        print(status)
    audio_q.put(indata.copy().reshape(-1))

# Find valid input device (default can be -1 in Docker/Colab)
default_input = None
for i in range(16):
    try:
        info = sd.query_devices(i)
        if info.get('max_input_channels', 0) > 0:
            default_input = i
            print(f'Input device [{i}]: {info["name"]}')
            break
    except (sd.PortAudioError, OSError):
        continue
if default_input is None:
    print('No microphone. (Use Section 9 with an audio file instead.)')

print('Listening... (Stop cell to exit)')
print('-' * 50)

In [ ]:
if default_input is None:
    print('Skipping mic (no device). Use Section 9 with an audio file.')
else:
    try:
        with sd.InputStream(channels=1, samplerate=SAMPLE_RATE, callback=sd_callback,
                            blocksize=int(SAMPLE_RATE * CHUNK_SEC), dtype='float32',
                            device=default_input):
            while True:
                audio_block = audio_q.get()
                chunk = buf.add(audio_block)
                if chunk is None:
                    continue

                if device == 'cuda':
                    torch.cuda.synchronize()
                start = time.time()

                logits, encoded_len, _ = asr_model.forward(
                    input_signal=torch.tensor(chunk).unsqueeze(0).to(device),
                    input_signal_length=torch.tensor([len(chunk)]).to(device),
                )

                if device == 'cuda':
                    torch.cuda.synchronize()
                lat_ms = (time.time() - start) * 1000

                preds = asr_model.decoding.rnnt_decoder_predictions_tensor(
                    logits=logits, encoded_lengths=encoded_len
                )
                text = preds[0] if preds else ''
                print(f'[+{lat_ms:.0f} ms] {text}')
    except sd.PortAudioError as e:
        print(f'Microphone error: {e}')
        print('(No mic in Docker/Colab. Use Section 9 with an audio file.)')
    except KeyboardInterrupt:
        print('\nStopped.')

## 9. Optional — Your Own Audio File

In [ ]:
YOUR_AUDIO = None  # e.g. 'my_recording.wav'
DEBUG_VAD = True  # Set True to see VAD-detected speech segments

if YOUR_AUDIO and os.path.exists(YOUR_AUDIO):
    from stt_optimized import load_silero_vad, get_speech_segments, streaming_transcribe_optimized
    
    your_audio, _ = librosa.load(YOUR_AUDIO, sr=SAMPLE_RATE, mono=True)
    duration = len(your_audio) / SAMPLE_RATE
    print(f'Loaded: {YOUR_AUDIO} ({duration:.2f}s)')
    
    vad_model, vad_utils = load_silero_vad()
    get_speech_ts = vad_utils[0]
    
    # VAD debug: show detected speech segments
    if DEBUG_VAD:
        segments = get_speech_segments(your_audio, SAMPLE_RATE, vad_model, get_speech_ts)
        print(f'\nVAD detected {len(segments)} speech segment(s):')
        for i, (start, end) in enumerate(segments):
            print(f'  [{i}] {start/SAMPLE_RATE:.2f}s - {end/SAMPLE_RATE:.2f}s  ({end-start} samples)')
        print()
    
    text, lat, ttfw = streaming_transcribe_optimized(
        your_audio, asr_model, SAMPLE_RATE,
        first_chunk_s=1.0, rest_chunk_s=2.0, use_vad=True,
        vad_model=vad_model, get_speech_timestamps=get_speech_ts, device=device,
    )
    print(f'\nTranscript: "{text}"')
    print(f'TTFW: {ttfw:.0f} ms  |  Avg latency: {np.mean(lat):.0f} ms')
else:
    print('Set YOUR_AUDIO to a valid path to test your own file.')